# Synthetic defects
> Generate synthetic defects

In [ ]:
#| default_exp patching.defect_synthesis

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#| export
import sys
from pathlib import Path


In [ ]:
#| export
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(CV_TOOLS))


In [ ]:
#| export
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
sys.path.append(str(custom_lib_path))


In [ ]:
#| export
from cv_tools.imports import *
from cv_tools.core import *
from dotenv import load_dotenv
from fastcore.all import *
from typing import List, Tuple, Optional
import random


In [ ]:
#| export
load_dotenv(dotenv_path=f'/home/ai_sintercra/Users/goni/workspace/projects/git_data/new_test/new_test/.env')

False

In [ ]:
from platform import system
if system() == "Windows":
    data_path= Path(r'C:\Users\goni\workspace\projects\data\easy_end_test')
else:
    data_path= Path(r'/home/ai_easypid.work/data/projects/easy_end_test/ET4/2B_solder_new_model/2B_solder_new_model_unzip/')

im_path = Path(data_path, 'main_im2_cropped_sn_images_deduped')
msk_path  = Path(data_path, 'label_studio_masks')
output_path = Path(data_path, 'synthetic_images_test')
print('Image path: ', im_path.exists())
print('Mask path: ', msk_path.exists())
images = list(Path(im_path).glob('*.png'))
image_names = get_name_(images)
print(f'Found {len(images)} images')


Image path:  True
Mask path:  True
Found 19 images


In [ ]:
#| export
class SemiconductorImageSynthesizer:
    def __init__(self, base_path: str):
        self.base_path = Path(base_path)
        self.fg_bank_path = self.base_path / "foreground_bank"
        self.bg_bank_path = self.base_path / "background_bank"
        self.synthetic_path = self.base_path / "synthetic_images"
        self.masks_path = self.base_path / "synthetic_masks"
        
        # Create directories
        for path in [self.fg_bank_path, self.bg_bank_path, self.synthetic_path, self.masks_path]:
            path.mkdir(parents=True, exist_ok=True)

In [ ]:
#| export
@patch
def generate_realistic_rectangular_dirt_contamination(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # input pin image
    mask: np.ndarray, # pin mask
    contamination_level: float = 0.4 # contamination intensity
    ) -> Tuple[np.ndarray, np.ndarray]:
    """Generate realistic dirt contamination constrained to rectangular regions"""
    
    result = image.copy().astype(np.float32)
    dirt_mask = mask.copy()
    h, w = image.shape[:2]
    
    # Generate rectangular defect mask
    defect_shape = random.choice(['rectangle', 'rotated_rectangle'])
    rect_mask = self.generate_rectangular_defect_mask(image.shape, mask, defect_shape)
    
    # Only apply dirt within rectangular region
    rect_coords = np.where(rect_mask > 127)
    if len(rect_coords[0]) == 0:
        return image, mask
    
    # Create realistic organic dirt pattern within rectangle
    # Use Perlin-like noise for natural contamination
    y_coords, x_coords = np.ogrid[:h, :w]
    
    # Generate multiple dirt clusters within rectangle
    num_dirt_spots = random.randint(3, 8)
    
    for _ in range(num_dirt_spots):
        # Random center within rectangular region
        rect_idx = random.randint(0, len(rect_coords[0]) - 1)
        center_y, center_x = rect_coords[0][rect_idx], rect_coords[1][rect_idx]
        
        # Create organic dirt blob
        blob_size = random.randint(5, min(h, w) // 8)
        dist = np.sqrt((y_coords - center_y)**2 + (x_coords - center_x)**2)
        dirt_blob = np.exp(-dist / blob_size)
        
        # Add organic irregularity using multiple frequency noise
        angle_map = np.arctan2(y_coords - center_y, x_coords - center_x)
        organic_shape = (1 + 0.4 * np.sin(angle_map * random.randint(3, 6)) + 
                        0.2 * np.sin(angle_map * random.randint(7, 12)))
        dirt_blob *= organic_shape
        
        # Apply only within rectangular mask
        dirt_area = (dirt_blob > 0.2) & (rect_mask > 127)
        
        if np.any(dirt_area):
            dirt_intensity = dirt_blob[dirt_area] * contamination_level
            
            # Realistic dirt effects - darken and add texture
            result[dirt_area] *= (1 - dirt_intensity * 0.5)  # Darken
            
            # Add brownish tint (in grayscale: brightness variation)
            brown_tint = np.random.normal(-15, 8, np.sum(dirt_area))
            result[dirt_area] += brown_tint * dirt_intensity
            
            # Add granular texture
            granular_noise = np.random.normal(0, 12, np.sum(dirt_area))
            result[dirt_area] += granular_noise * dirt_intensity
            
            # Update mask to include dirt region
            dirt_mask = np.maximum(dirt_mask, dirt_area.astype(np.uint8) * 255)
    
    return np.clip(result, 0, 255).astype(np.uint8), dirt_mask


In [ ]:
#| export
@patch
def generate_realistic_rectangular_deep_shadow(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # input pin image
    mask: np.ndarray, # pin mask
    shadow_intensity: float = 0.8 # shadow darkness (0.5-0.9 for deep shadows)
    ) -> Tuple[np.ndarray, np.ndarray]:
    """Generate realistic deep shadows that can completely hide pin parts in rectangular regions"""
    
    result = image.copy().astype(np.float32)
    shadow_mask = mask.copy()
    h, w = image.shape[:2]
    
    # Generate rectangular shadow region
    defect_shape = random.choice(['rectangle', 'rotated_rectangle'])
    rect_mask = self.generate_rectangular_defect_mask(image.shape, mask, defect_shape)
    
    # Only apply shadow within rectangular region
    rect_coords = np.where(rect_mask > 127)
    if len(rect_coords[0]) == 0:
        return image, mask
    
    # Create realistic shadow gradient within rectangle
    y_coords, x_coords = np.ogrid[:h, :w]
    
    # Determine shadow direction (lighting angle)
    shadow_types = ['top_down', 'side_cast', 'diagonal_harsh']
    shadow_type = random.choice(shadow_types)
    
    if shadow_type == 'top_down':
        # Vertical shadow gradient - darker at top
        rect_y_coords = rect_coords[0]
        min_y, max_y = rect_y_coords.min(), rect_y_coords.max()
        if max_y > min_y:
            shadow_gradient = ((rect_y_coords - min_y) / (max_y - min_y)) * shadow_intensity
        else:
            shadow_gradient = np.full(len(rect_coords[0]), shadow_intensity)
            
    elif shadow_type == 'side_cast':
        # Horizontal shadow gradient
        rect_x_coords = rect_coords[1]
        min_x, max_x = rect_x_coords.min(), rect_x_coords.max()
        if max_x > min_x:
            shadow_gradient = ((rect_x_coords - min_x) / (max_x - min_x)) * shadow_intensity
        else:
            shadow_gradient = np.full(len(rect_coords[0]), shadow_intensity)
            
    else:  # diagonal_harsh
        # Diagonal shadow with harsh transition
        rect_center_y = rect_coords[0].mean()
        rect_center_x = rect_coords[1].mean()
        
        # Random diagonal direction
        angle = random.uniform(0, 2 * np.pi)
        shadow_dir_x = np.cos(angle)
        shadow_dir_y = np.sin(angle)
        
        # Project coordinates onto shadow direction
        projected = ((rect_coords[0] - rect_center_y) * shadow_dir_y + 
                    (rect_coords[1] - rect_center_x) * shadow_dir_x)
        
        if projected.max() > projected.min():
            proj_norm = (projected - projected.min()) / (projected.max() - projected.min())
            shadow_gradient = proj_norm * shadow_intensity
        else:
            shadow_gradient = np.full(len(rect_coords[0]), shadow_intensity)
    
    # Apply deep shadow effects
    # Create areas of complete darkness (pin parts completely hidden)
    deep_shadow_threshold = 0.7  # Areas with shadow > 0.7 become very dark
    complete_darkness_threshold = 0.85  # Areas with shadow > 0.85 become nearly black
    
    # Apply shadow with realistic falloff
    shadow_factor = 1 - shadow_gradient
    
    # Areas in deep shadow
    deep_shadow_areas = shadow_gradient > deep_shadow_threshold
    complete_darkness_areas = shadow_gradient > complete_darkness_threshold
    
    # Apply shadow effects
    result[rect_coords] *= shadow_factor
    
    # Deep shadow areas - very dark with subtle texture
    if np.any(deep_shadow_areas):
        deep_indices = rect_coords[0][deep_shadow_areas], rect_coords[1][deep_shadow_areas]
        
        # Make very dark but not completely black
        result[deep_indices] *= 0.15  # 85% darkness
        
        # Add subtle shadow texture
        shadow_texture = np.random.normal(0, 3, np.sum(deep_shadow_areas))
        result[deep_indices] += shadow_texture
    
    # Complete darkness areas - pin parts completely hidden
    if np.any(complete_darkness_areas):
        dark_indices = rect_coords[0][complete_darkness_areas], rect_coords[1][complete_darkness_areas]
        
        # Nearly black - pin details completely lost
        result[dark_indices] *= 0.05  # 95% darkness
        
        # Minimal texture in darkness
        dark_texture = np.random.normal(0, 1, np.sum(complete_darkness_areas))
        result[dark_indices] += dark_texture
        
        # Update mask - completely dark areas might be considered background
        # But keep them in mask for training purposes
        shadow_mask[dark_indices] = 128  # Intermediate value to indicate shadowed pin area
    
    # Add realistic shadow edge softening
    # Create soft transitions at shadow boundaries
    kernel = np.ones((3, 3), np.uint8) / 9
    shadow_blur = cv2.filter2D(result, -1, kernel)
    
    # Blend at shadow edges
    edge_softening = random.uniform(0.1, 0.3)
    result = (1 - edge_softening) * result + edge_softening * shadow_blur
    
    return np.clip(result, 0, 255).astype(np.uint8), shadow_mask


In [ ]:
#| export
@patch
def generate_realistic_rectangular_oxidation_corrosion(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # input pin image
    mask: np.ndarray, # pin mask
    severity: float = 0.4 # oxidation severity
    ) -> Tuple[np.ndarray, np.ndarray]:
    """Generate realistic oxidation/corrosion patterns constrained to rectangular regions"""
    
    result = image.copy().astype(np.float32)
    corroded_mask = mask.copy()
    h, w = image.shape[:2]
    
    # Generate rectangular defect mask
    defect_shape = random.choice(['rectangle', 'rotated_rectangle'])
    rect_mask = self.generate_rectangular_defect_mask(image.shape, mask, defect_shape)
    
    # Only apply corrosion within rectangular region
    rect_coords = np.where(rect_mask > 127)
    if len(rect_coords[0]) == 0:
        return image, mask
    
    # Create realistic corrosion pattern starting from edges of rectangle
    y_coords, x_coords = np.ogrid[:h, :w]
    
    # Find edges of rectangular region for corrosion initiation
    kernel = np.ones((3, 3), np.uint8)
    rect_edges = cv2.morphologyEx(rect_mask, cv2.MORPH_GRADIENT, kernel)
    edge_coords = np.where(rect_edges > 127)
    
    if len(edge_coords[0]) == 0:
        return image, mask
    
    # Generate multiple corrosion centers along rectangle edges
    num_corrosion_spots = random.randint(2, 5)
    
    for _ in range(num_corrosion_spots):
        # Pick random edge point as corrosion center
        edge_idx = random.randint(0, len(edge_coords[0]) - 1)
        center_y, center_x = edge_coords[0][edge_idx], edge_coords[1][edge_idx]
        
        # Create corrosion spreading pattern
        dist_from_center = np.sqrt((y_coords - center_y)**2 + (x_coords - center_x)**2)
        
        # Corrosion spreads inward from edges with decreasing intensity
        max_spread = min(h, w) // 6 * severity
        corrosion_intensity = np.exp(-dist_from_center / max_spread)
        
        # Add organic growth pattern using multiple frequencies
        angle_map = np.arctan2(y_coords - center_y, x_coords - center_x)
        organic_growth = (1 + 0.5 * np.sin(angle_map * random.randint(4, 8)) + 
                         0.3 * np.sin(angle_map * random.randint(9, 15)))
        corrosion_intensity *= organic_growth
        
        # Add randomness for realistic corrosion texture
        noise = np.random.random((h, w)) * 0.7 + 0.3
        corrosion_pattern = corrosion_intensity * noise
        
        # Apply only within rectangular mask
        corrosion_area = (corrosion_pattern > 0.25) & (rect_mask > 127)
        
        if np.any(corrosion_area):
            corr_intensity = corrosion_pattern[corrosion_area] * severity
            
            # Realistic corrosion effects
            # Darken corroded areas (oxidation typically darkens metal)
            result[corrosion_area] *= (1 - corr_intensity * 0.6)
            
            # Add brownish/reddish tint (rust color in grayscale)
            rust_tint = np.random.normal(-20, 12, np.sum(corrosion_area))
            result[corrosion_area] += rust_tint * corr_intensity
            
            # Add surface roughness and pitting
            surface_roughness = np.random.normal(0, 15, np.sum(corrosion_area))
            result[corrosion_area] += surface_roughness * corr_intensity
            
            # Create pitting effect (small dark spots)
            pitting_prob = corr_intensity * 0.3
            pitting_mask = np.random.random(np.sum(corrosion_area)) < pitting_prob
            if np.any(pitting_mask):
                pit_coords = np.where(corrosion_area)[0][pitting_mask], np.where(corrosion_area)[1][pitting_mask]
                result[pit_coords] *= 0.4  # Deep pits are very dark
            
            # Expand mask to include corroded area
            kernel_small = np.ones((2, 2), np.uint8)
            expanded_corrosion = cv2.dilate(corrosion_area.astype(np.uint8) * 255, kernel_small)
            corroded_mask = np.maximum(corroded_mask, expanded_corrosion)
    
    return np.clip(result, 0, 255).astype(np.uint8), corroded_mask


In [ ]:
#| export
@patch
def generate_realistic_rectangular_flux_residue(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # input pin image
    mask: np.ndarray, # pin mask
    residue_amount: float = 0.3 # flux residue amount
    ) -> Tuple[np.ndarray, np.ndarray]:
    """Generate realistic flux residue patterns constrained to rectangular regions"""
    
    result = image.copy().astype(np.float32)
    residue_mask = mask.copy()
    h, w = image.shape[:2]
    
    # Generate rectangular defect mask - flux residue typically in larger patches
    defect_shape = random.choices(
        ['rectangle', 'rotated_rectangle'], 
        weights=[0.8, 0.2]  # Prefer regular rectangles for flux
    )[0]
    rect_mask = self.generate_rectangular_defect_mask(image.shape, mask, defect_shape)
    
    # Only apply flux within rectangular region
    rect_coords = np.where(rect_mask > 127)
    if len(rect_coords[0]) == 0:
        return image, mask
    
    # Create realistic flux residue pattern within rectangle
    y_coords, x_coords = np.ogrid[:h, :w]
    
    # Generate multiple flux blobs within rectangle
    num_flux_areas = random.randint(2, 6)
    
    for _ in range(num_flux_areas):
        # Random center within rectangular region
        rect_idx = random.randint(0, len(rect_coords[0]) - 1)
        center_y, center_x = rect_coords[0][rect_idx], rect_coords[1][rect_idx]
        
        # Create organic flux blob
        blob_size = random.randint(8, min(h, w) // 4)
        dist = np.sqrt((y_coords - center_y)**2 + (x_coords - center_x)**2)
        flux_blob = np.exp(-dist / blob_size)
        
        # Add realistic flux flow pattern
        # Flux tends to flow and create streaky patterns
        angle_map = np.arctan2(y_coords - center_y, x_coords - center_x)
        flow_pattern = (1 + 0.6 * np.sin(angle_map * random.randint(2, 4)) + 
                       0.4 * np.cos(angle_map * random.randint(5, 8)))
        flux_blob *= flow_pattern
        
        # Add surface tension effects (flux beading)
        surface_tension = np.random.random((h, w)) * 0.8 + 0.2
        flux_blob *= surface_tension
        
        # Apply only within rectangular mask
        flux_area = (flux_blob > 0.3) & (rect_mask > 127)
        
        if np.any(flux_area):
            flux_intensity = flux_blob[flux_area] * residue_amount
            
            # Realistic flux residue effects
            # Flux can be lighter or darker than base material
            flux_type = random.choice(['light_flux', 'dark_flux', 'mixed_flux'])
            
            if flux_type == 'light_flux':
                # Light, glossy flux residue
                result[flux_area] *= (1 + flux_intensity * 0.4)
                
                # Add glossy reflection spots
                gloss_spots = np.random.random(np.sum(flux_area)) < flux_intensity * 0.2
                if np.any(gloss_spots):
                    gloss_coords = np.where(flux_area)[0][gloss_spots], np.where(flux_area)[1][gloss_spots]
                    result[gloss_coords] += 30 * flux_intensity[gloss_spots]
                    
            elif flux_type == 'dark_flux':
                # Dark, burnt flux residue
                result[flux_area] *= (1 - flux_intensity * 0.5)
                
                # Add carbonized spots
                burnt_spots = np.random.random(np.sum(flux_area)) < flux_intensity * 0.15
                if np.any(burnt_spots):
                    burnt_coords = np.where(flux_area)[0][burnt_spots], np.where(flux_area)[1][burnt_spots]
                    result[burnt_coords] *= 0.3  # Very dark burnt areas
                    
            else:  # mixed_flux
                # Mixed light and dark flux
                light_areas = np.random.random(np.sum(flux_area)) > 0.5
                result[flux_area] = np.where(
                    light_areas,
                    result[flux_area] * (1 + flux_intensity * 0.3),
                    result[flux_area] * (1 - flux_intensity * 0.3)
                )
            
            # Add flux texture and surface irregularities
            flux_texture = np.random.normal(0, 8, np.sum(flux_area))
            result[flux_area] += flux_texture * flux_intensity
            
            # Update mask to include flux area
            residue_mask = np.maximum(residue_mask, flux_area.astype(np.uint8) * 255)
    
    return np.clip(result, 0, 255).astype(np.uint8), residue_mask


In [ ]:
#| export
@patch
def generate_realistic_rectangular_surface_scratches(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # input pin image
    mask: np.ndarray, # pin mask
    scratch_intensity: float = 0.4 # scratch visibility
    ) -> np.ndarray:
    """Generate realistic surface scratches constrained to rectangular regions"""
    
    result = image.copy().astype(np.float32)
    h, w = image.shape[:2]
    
    # Generate rectangular defect mask
    defect_shape = random.choice(['rectangle', 'rotated_rectangle'])
    rect_mask = self.generate_rectangular_defect_mask(image.shape, mask, defect_shape)
    
    # Only apply scratches within rectangular region
    rect_coords = np.where(rect_mask > 127)
    if len(rect_coords[0]) == 0:
        return image
    
    # Generate realistic scratches within rectangle
    num_scratches = random.randint(2, 6)
    
    for _ in range(num_scratches):
        # Random scratch parameters within rectangular bounds
        rect_y_coords, rect_x_coords = rect_coords[0], rect_coords[1]
        min_y, max_y = rect_y_coords.min(), rect_y_coords.max()
        min_x, max_x = rect_x_coords.min(), rect_x_coords.max()
        
        # Random start point within rectangle
        start_y = random.randint(min_y, max_y)
        start_x = random.randint(min_x, max_x)
        
        # Ensure start point is within pin area
        if mask[start_y, start_x] <= 127:
            continue
            
        # Random scratch direction and length
        angle = random.uniform(0, 2 * np.pi)
        dx = np.cos(angle)
        dy = np.sin(angle)
        
        # Scratch length proportional to rectangle size
        rect_width = max_x - min_x
        rect_height = max_y - min_y
        max_scratch_length = min(rect_width, rect_height) // 2
        scratch_length = random.randint(max_scratch_length // 3, max_scratch_length)
        
        # Scratch width and depth
        scratch_width = random.randint(1, 3)
        scratch_depth = random.uniform(0.3, 0.8) * scratch_intensity
        
        # Create scratch path
        scratch_points = []
        for i in range(scratch_length):
            # Add slight randomness to scratch path for realism
            noise_x = random.uniform(-0.5, 0.5)
            noise_y = random.uniform(-0.5, 0.5)
            
            x = int(start_x + i * dx + noise_x)
            y = int(start_y + i * dy + noise_y)
            
            # Check bounds and ensure within rectangular region
            if (0 <= x < w and 0 <= y < h and 
                mask[y, x] > 127 and rect_mask[y, x] > 127):
                scratch_points.append((x, y))
        
        if not scratch_points:
            continue
            
        # Apply scratch effects along the path
        for x, y in scratch_points:
            # Apply scratch with varying width
            for dx_offset in range(-scratch_width, scratch_width + 1):
                for dy_offset in range(-scratch_width, scratch_width + 1):
                    nx, ny = x + dx_offset, y + dy_offset
                    
                    if (0 <= nx < w and 0 <= ny < h and 
                        mask[ny, nx] > 127 and rect_mask[ny, nx] > 127):
                        
                        # Distance from scratch center affects intensity
                        dist_from_center = np.sqrt(dx_offset**2 + dy_offset**2)
                        if dist_from_center <= scratch_width:
                            # Scratch intensity decreases with distance from center
                            intensity_falloff = 1 - (dist_from_center / scratch_width)
                            local_intensity = scratch_depth * intensity_falloff
                            
                            # Create scratch effect - darker line with bright edges
                            if dist_from_center < 0.5:  # Center of scratch - darkest
                                result[ny, nx] *= (1 - local_intensity * 0.8)
                            elif dist_from_center < 1.0:  # Edge of scratch - slight brightening
                                result[ny, nx] *= (1 + local_intensity * 0.2)
                            
                            # Add micro-texture to scratch
                            scratch_texture = np.random.normal(0, 5 * local_intensity)
                            result[ny, nx] += scratch_texture
    
    return np.clip(result, 0, 255).astype(np.uint8)


In [ ]:
#| export
@patch
def generate_rectangular_defect_mask(
    self:SemiconductorImageSynthesizer,
    image_shape: Tuple[int, int], # (height, width)
    pin_mask: np.ndarray, # original pin mask to constrain defect location
    defect_type: str = 'rectangle' # 'rectangle' or 'rotated_rectangle'
    ) -> np.ndarray:
    """Generate rectangular or rotated rectangular defect mask within pin area"""
    
    h, w = image_shape
    defect_mask = np.zeros((h, w), dtype=np.uint8)
    
    # Find pin area boundaries
    pin_coords = np.where(pin_mask > 127)
    if len(pin_coords[0]) == 0:
        return defect_mask
    
    # Get pin bounding box
    min_y, max_y = pin_coords[0].min(), pin_coords[0].max()
    min_x, max_x = pin_coords[1].min(), pin_coords[1].max()
    
    pin_width = max_x - min_x
    pin_height = max_y - min_y
    
    if defect_type == 'rectangle':
        # Generate axis-aligned rectangle
        # Defect size: 20-60% of pin dimensions
        defect_w = random.randint(int(pin_width * 0.2), int(pin_width * 0.6))
        defect_h = random.randint(int(pin_height * 0.2), int(pin_height * 0.6))
        
        # Random position within pin bounds
        start_x = random.randint(min_x, max(min_x + 1, max_x - defect_w))
        start_y = random.randint(min_y, max(min_y + 1, max_y - defect_h))
        
        # Draw rectangle
        defect_mask[start_y:start_y + defect_h, start_x:start_x + defect_w] = 255
        
    else:  # rotated_rectangle
        # Generate rotated rectangle using cv2.fillPoly
        # Center within pin area
        center_x = random.randint(min_x + pin_width//4, max_x - pin_width//4)
        center_y = random.randint(min_y + pin_height//4, max_y - pin_height//4)
        
        # Rectangle dimensions
        rect_width = random.randint(int(pin_width * 0.2), int(pin_width * 0.6))
        rect_height = random.randint(int(pin_height * 0.2), int(pin_height * 0.6))
        
        # Random rotation angle
        angle = random.uniform(0, 180)  # degrees
        
        # Create rotated rectangle points
        rect = ((center_x, center_y), (rect_width, rect_height), angle)
        box = cv2.boxPoints(rect)
        box = box.astype(np.int32)
        
        # Fill the rotated rectangle
        cv2.fillPoly(defect_mask, [box], 255)
    
    # Ensure defect is within pin area only
    defect_mask = cv2.bitwise_and(defect_mask, pin_mask)
    
    return defect_mask


In [ ]:
#| export
@patch
def generate_enhanced_realistic_rectangular_defective_variants(
    self:SemiconductorImageSynthesizer, 
    original_image: np.ndarray, # original pin image
    original_mask: np.ndarray, # original pin mask
    num_variants: int = 12, # number of defective variants to generate
    intensity_range: Tuple[float, float] = (0.5, 0.9) # defect intensity range (high intensity)
    ) -> List[Tuple[np.ndarray, np.ndarray, str]]:
    """Generate multiple defective variants with ENHANCED REALISTIC rectangular defects"""
    
    variants = []
    defect_types = ['dirt', 'deep_shadow', 'oxidation', 'flux', 'scratches']
    
    for i in range(num_variants):
        # Create variant with random defect combination
        variant_image = original_image.copy()
        variant_mask = original_mask.copy()
        applied_defects = []
        
        # Number of defect types to apply (1-3 types per variant)
        num_defects = random.randint(1, 3)
        selected_defects = random.sample(defect_types, num_defects)
        
        # Random intensity for this variant
        defect_intensity = random.uniform(*intensity_range)
        
        # Apply each selected defect type
        for defect_type in selected_defects:
            try:
                if defect_type == 'dirt':
                    variant_image, variant_mask = self.generate_realistic_rectangular_dirt_contamination(
                        variant_image, variant_mask, defect_intensity
                    )
                    applied_defects.append('dirt')
                    
                elif defect_type == 'deep_shadow':
                    # Use higher intensity for deep shadows to ensure pin parts are hidden
                    shadow_intensity = random.uniform(0.7, 0.95)
                    variant_image, variant_mask = self.generate_realistic_rectangular_deep_shadow(
                        variant_image, variant_mask, shadow_intensity
                    )
                    applied_defects.append('deep_shadow')
                    
                elif defect_type == 'oxidation':
                    variant_image, variant_mask = self.generate_realistic_rectangular_oxidation_corrosion(
                        variant_image, variant_mask, defect_intensity
                    )
                    applied_defects.append('oxidation')
                    
                elif defect_type == 'flux':
                    variant_image, variant_mask = self.generate_realistic_rectangular_flux_residue(
                        variant_image, variant_mask, defect_intensity
                    )
                    applied_defects.append('flux')
                    
                elif defect_type == 'scratches':
                    variant_image = self.generate_realistic_rectangular_surface_scratches(
                        variant_image, variant_mask, defect_intensity
                    )
                    applied_defects.append('scratches')
                    
            except Exception as e:
                print(f"Warning: Failed to apply {defect_type} defect: {e}")
                continue
        
        # Create descriptive filename
        if applied_defects:
            defect_description = "_".join(applied_defects)
            filename_suffix = f"enhanced_realistic_rect_{defect_description}_v{i+1:02d}"
        else:
            filename_suffix = f"enhanced_realistic_rect_clean_v{i+1:02d}"
        
        variants.append((variant_image, variant_mask, filename_suffix))
    
    return variants


In [ ]:
#| export
def process_enhanced_realistic_rectangular_defects(
    image_dir: str, 
    mask_dir: str,
    output_dir: str,
    variants_per_image: int = 12
    ):
    """
    Process dataset with ENHANCED REALISTIC rectangular defects
    
    🔧 KEY FEATURES:
    - Realistic dirt contamination (organic patterns in rectangular regions)
    - Deep shadows that completely hide pin parts (rectangular shadow areas)  
    - Realistic oxidation/corrosion (edge-based spreading in rectangles)
    - Realistic flux residue (flow patterns in rectangular patches)
    - Realistic surface scratches (mechanical damage in rectangular zones)
    
    🎯 ALL DEFECTS ARE CONSTRAINED TO RECTANGULAR/ROTATED RECTANGULAR SHAPES
    """
    
    synthesizer = SemiconductorImageSynthesizer(
        base_path=str(output_dir)
    )
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Create output directories
    os.makedirs(f"{output_dir}/images", exist_ok=True)
    os.makedirs(f"{output_dir}/masks", exist_ok=True)
    
    # Get all image files
    image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    total_generated = 0
    
    print("🚀 PROCESSING ENHANCED REALISTIC RECTANGULAR DEFECTS")
    print("=" * 60)
    
    for img_file in image_files:
        try:
            # Load original image and mask
            img_path = os.path.join(image_dir, img_file)
            mask_file = img_file.replace('.jpg', '.png').replace('.jpeg', '.png')
            mask_path = os.path.join(mask_dir, mask_file)
            
            if not os.path.exists(mask_path):
                print(f"⚠️  Mask not found for {img_file}, skipping...")
                continue
                
            original_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            original_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            
            if original_img is None or original_mask is None:
                print(f"⚠️  Failed to load {img_file}, skipping...")
                continue
            
            print(f"📸 Processing: {img_file}")
            
            # Generate enhanced realistic rectangular defective variants
            variants = synthesizer.generate_enhanced_realistic_rectangular_defective_variants(
                original_img, 
                original_mask, 
                num_variants=variants_per_image,
                intensity_range=(0.9, .99)  # High intensity for visible defects
            )
            
            # Save all variants
            base_name = os.path.splitext(img_file)[0]
            
            for variant_img, variant_mask, suffix in variants:
                # Save defective image
                img_output_name = f"{base_name}_{suffix}.png"
                mask_output_name = f"{base_name}_{suffix}.png"
                
                img_output_path = os.path.join(f"{output_dir}/images", img_output_name)
                mask_output_path = os.path.join(f"{output_dir}/masks", mask_output_name)
                
                cv2.imwrite(img_output_path, variant_img)
                cv2.imwrite(mask_output_path, variant_mask)
                
                total_generated += 1
            
            print(f"   ✅ Generated {len(variants)} variants")
            
        except Exception as e:
            print(f"❌ Error processing {img_file}: {e}")
            continue
    
    print("=" * 60)
    print(f"🎯 ENHANCED REALISTIC RECTANGULAR DEFECT GENERATION COMPLETE!")
    print(f"   📊 Total defective variants generated: {total_generated}")
    print(f"   📁 Output directory: {output_dir}")
    print("=" * 60)
    print("🔧 DEFECT FEATURES:")
    print("   • Organic dirt patterns in rectangular regions")
    print("   • Deep shadows that completely hide pin parts")
    print("   • Edge-based corrosion spreading in rectangles") 
    print("   • Realistic flux flow patterns in rectangular patches")
    print("   • Mechanical scratches in rectangular zones")
    print("   • ALL defects constrained to rectangular/rotated rectangular shapes")
    print("=" * 60)


In [ ]:
process_enhanced_realistic_rectangular_defects(
    image_dir=str(im_path),
    mask_dir=str(msk_path),
    output_dir=str(output_path),
    variants_per_image=10
)

🚀 PROCESSING ENHANCED REALISTIC RECTANGULAR DEFECTS
📸 Processing: VFV4.9.0.3_2025060418244066_Out_138_Missing Lead_image2.png
   ✅ Generated 10 variants
📸 Processing: VFV4.9.0.3_2025060418324293_Out_138_Missing Lead_image2_2.png
   ✅ Generated 10 variants
📸 Processing: VFV4.9.0.3_2025060419025109_Out_138_Missing Lead_image2.png
   ✅ Generated 10 variants
📸 Processing: VFV4.9.0.3_2025060419241981_Out_138_Missing Lead_image2_1.png
   ✅ Generated 10 variants
📸 Processing: VFV4.9.0.3_2025060419243230_Out_138_Missing Lead_image2.png
   ✅ Generated 10 variants
📸 Processing: VFV4.9.0.3_2025060419341510_Out_138_Missing Lead_image2_1.png
   ✅ Generated 10 variants
📸 Processing: VFV4.9.0.3_2025060419400625_Out_138_Missing Lead_image2_3.png
   ✅ Generated 10 variants
📸 Processing: VFV4.9.0.3_2025060419512628_Out_138_Missing Lead_image2.png
   ✅ Generated 10 variants
📸 Processing: VFV4.9.0.3_2025060420005357_Out_138_Missing Lead_image2.png
   ✅ Generated 10 variants
📸 Processing: VFV4.9.0.3_202506

In [ ]:
#| export
CURRETNT_NB='/home/ai_sintercra/Users/goni/workspace/projects/git_data/new_test/nbs'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('56_patching.defect_synthesis.ipynb')